# Polymorphic fields

A field is **polymorphic** ("multiform") when its value is not always the same
JSON type across records -- a `quantity` that is `"35 nm"` (string) in one
record, `35` (number) in another, `{"value": 35, "unit": "nm"}` (object) in a
third.

JSON Schema lets you declare this with a list-valued `type`, and this evaluator
supports it directly:

- `{"type": ["string", "null"]}` -- nullable; the `null` is dropped, so it is
  just a `string` field.
- `{"type": ["string", "number", "object"]}` -- a genuine multi-shape field.

Existing schemas (e.g. generated by pydantic) usually express this as `anyOf`
instead -- `resolve_schema_references()` converts that to the list-valued form,
see the last section below.


## The problem: the default comparator is coarse

Declare `quantity` honestly as `["string", "number", "object"]`, you need a custom comparator to deal with the different types.

In [2]:
from struct_extract_eval import annotate_xeval, evaluate
from struct_extract_eval.core.comparators.comparator import ComparatorResult
from struct_extract_eval.core.comparators.registry import register as register_comparator


def show(title, result):
    print(title)
    for rec in result.records:
        for fr in rec.field_results:
            print(f"  [{rec.record_id}] {fr.path:<8} gold={fr.gold_value!r:<24} extracted={fr.extracted_value!r:<20} -> {fr.status}")
    print(f"  mean F1={result.mean_f1:.2f}")


GOLD = [
    {"quantity": {"value": 35, "unit": "nm"}},  # object form
    {"quantity": 50},                            # number form
    {"quantity": "60 nm"},                       # string form
]
EXTRACTED = [
    {"quantity": "35 nm"},   # same meaning as gold, different shape
    {"quantity": "50"},      # same meaning as gold, different shape
    {"quantity": "65 nm"},   # genuinely wrong (60 != 65)
]

# Honest multi-type declaration. No comparator -> default `exact`.
# annotate_xeval assigns that default (evaluate does not annotate for you).
schema_default = {
    "type": "object",
    "properties": {"quantity": {"type": ["string", "number", "object"]}},
}
annotate_xeval(schema_default)

show("default exact comparator", evaluate(GOLD, EXTRACTED, schema_default))


default exact comparator
  [0] quantity gold={'value': 35, 'unit': 'nm'} extracted='35 nm'              -> mismatch
  [1] quantity gold=50                       extracted='50'                 -> mismatch
  [2] quantity gold='60 nm'                  extracted='65 nm'              -> mismatch
  mean F1=0.00


In [3]:
def _to_value_unit(v):
    if isinstance(v, dict):
        return v.get("value"), v.get("unit")
    if isinstance(v, (int, float)):
        return float(v), None
    if isinstance(v, str):
        parts = v.split()
        try:
            value = float(parts[0]) if parts else None
        except ValueError:
            # not a number (e.g. "n/a"): compare as raw text, never crash the run
            return v.strip(), None
        unit = parts[1] if len(parts) > 1 else None
        return value, unit
    return None, None


def compare_quantity(gold, extracted, params):
    if _to_value_unit(gold) == _to_value_unit(extracted):
        return ComparatorResult(score=1.0, comparator="quantity")
    return ComparatorResult(score=0.0, comparator="quantity", reason=f"{gold!r} != {extracted!r}")


register_comparator("quantity", compare_quantity, overwrite=True)

schema_poly = {
    "type": "object",
    "properties": {
        "quantity": {"type": ["string", "number", "object"], "x-eval-compare": "quantity"},
    },
}
annotate_xeval(schema_poly)   # no-op for `quantity` (it already has a comparator)

show("with quantity comparator", evaluate(GOLD, EXTRACTED, schema_poly))

with quantity comparator
  [0] quantity gold={'value': 35, 'unit': 'nm'} extracted='35 nm'              -> match
  [1] quantity gold=50                       extracted='50'                 -> match
  [2] quantity gold='60 nm'                  extracted='65 nm'              -> mismatch
  mean F1=0.67


Now the two cross-shape-but-equal records match, and the genuinely wrong one
(`60 nm` vs `65 nm`) is still a mismatch. One comparator, every shape covered.

## Getting the multi-type declaration from an existing JSON Schema

You rarely hand-write `type: ["string", "number", "object"]`. When the schema
comes from somewhere else -- e.g. pydantic, where `str | float | Measurement`
becomes `anyOf: [{"$ref": ...}, {"type": "string"}, {"type": "number"}]` --
`resolve_schema_references()` collapses that `anyOf` into the same list-valued
`type`:

- `{"type": "null"}` branches are dropped (nullable is handled by value
  presence, not type).
- Branch structure (the `properties` of `Measurement` here) is dropped with a
  warning: a multi-type field is scored as one unit by its comparator, never
  structurally.
- Not collapsed: `oneOf`, `if`-`then`-`else`, and an `anyOf` whose branches
  share a single type (e.g. two different object shapes) -- a type list cannot
  distinguish those.

In [4]:
from struct_extract_eval import resolve_schema_references

# What pydantic generates for:  quantity: Measurement | str | float | None
raw_schema = {
    "$defs": {
        "Measurement": {
            "type": "object",
            "properties": {
                "value": {"type": "number"},
                "unit": {"type": "string"},
            },
        },
    },
    "type": "object",
    "properties": {
        "quantity": {
            "anyOf": [
                {"$ref": "#/$defs/Measurement"},
                {"type": "string"},
                {"type": "number"},
                {"type": "null"},
            ],
        },
    },
}

schema_resolved = resolve_schema_references(raw_schema)
print(schema_resolved["properties"]["quantity"])

# Same field as before, so reuse the registered `quantity` comparator.
schema_resolved["properties"]["quantity"]["x-eval-compare"] = "quantity"
annotate_xeval(schema_resolved)

show("resolved from anyOf", evaluate(GOLD, EXTRACTED, schema_resolved))

resolve_schema_references handles $ref, allOf, anyOf[type, null], and anyOf
with multiple non-null types (collapsed to a list-valued `type` -- a
comparator-owned multi-type field; branch properties/items are dropped).
The following are NOT handled:
  - oneOf: type info lost, field has no 'type' key -> SchemaError at parse time
  - anyOf whose branches share one type (e.g. two object shapes) or lack a
    'type' key: anyOf is kept -> SchemaError at parse time
  - if/then/else: conditional properties lost -- may cause SchemaError or silently miss fields
For schemas with these keywords, use infer_schema(instances) instead.
anyOf at 'properties.quantity' collapsed to multi-type ['object', 'string', 'number']; branch keys ['properties'] were dropped. The field is scored as one unit by its comparator (assign x-eval-compare), not structurally.


{'type': ['object', 'string', 'number']}
resolved from anyOf
  [0] quantity gold={'value': 35, 'unit': 'nm'} extracted='35 nm'              -> match
  [1] quantity gold=50                       extracted='50'                 -> match
  [2] quantity gold='60 nm'                  extracted='65 nm'              -> mismatch
  mean F1=0.67


## Validating polymorphic gold

By default `validate_gold` treats `type` as a hint (mismatches only warn). For a
multi-type field it accepts gold of any declared shape. If you want to *enforce*
that gold is one of the declared types, pass `strict_types=True`:

```python
from struct_extract_eval import validate_gold

validate_gold(GOLD, schema_poly)
validate_gold(GOLD, schema_poly, strict_types=True)  # raises GoldValidationError if a gold value is not one of string/number/object (null is exempt)
```